# LMSYS Chatbot Arena — Inference + Submission

Load fine-tuned Gemma2-9B adapter, run inference on test set, generate `submission.csv`.  
**Must run on Kaggle with GPU, internet disabled, ≤9 hours.**

### Required Kaggle Datasets (add as Input):
- `lmsys-chatbot-arena` (competition data — for test.csv)
- `gemma-2-9b-it` (base model weights)
- Your trained adapter output (from training notebook)

In [ ]:
# ============================================================
# CONFIG
# ============================================================
class CFG:
    # Paths — adjust adapter_path to your saved dataset
    model_path = '/kaggle/input/gemma-2-2b-it'
    adapter_path = '/kaggle/input/lmsys-trained-adapter/lora-adapter'  # <-- your adapter dataset
    data_path = '/kaggle/input/lmsys-chatbot-arena'
    
    # Inference config
    max_length = 1536
    batch_size = 1  # keep at 1 for T4 memory
    num_labels = 3
    
    # TTA — swap response_a and response_b, average predictions
    use_tta = True
    
    # Quantization for inference (4-bit matches QLoRA training)
    use_4bit = True

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
)
from peft import PeftModel
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

## 1. Load Test Data

In [ ]:
test_df = pd.read_csv(f'{CFG.data_path}/test.csv')
sub_df = pd.read_csv(f'{CFG.data_path}/sample_submission.csv')
print(f'Test: {len(test_df)} rows')
print(f'Columns: {list(test_df.columns)}')

In [ ]:
class PreferenceDataset(Dataset):
    """Dataset for inference — supports optional TTA (response swap)."""
    
    def __init__(self, df, tokenizer, max_length, swap=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.swap = swap  # if True, swap response_a and response_b
    
    def __len__(self):
        return len(self.df)
    
    def _format_conversation(self, row):
        prompts = json.loads(row['prompt'])
        
        if self.swap:
            responses_a = json.loads(row['response_b'])  # swapped
            responses_b = json.loads(row['response_a'])  # swapped
        else:
            responses_a = json.loads(row['response_a'])
            responses_b = json.loads(row['response_b'])
        
        rounds = []
        for i in range(len(prompts)):
            r_a = responses_a[i] if i < len(responses_a) else ''
            r_b = responses_b[i] if i < len(responses_b) else ''
            rounds.append(
                f'<prompt>\n{prompts[i]}\n</prompt>\n'
                f'<response_a>\n{r_a}\n</response_a>\n'
                f'<response_b>\n{r_b}\n</response_b>'
            )
        
        # Left truncation: keep later turns
        for k in range(len(rounds)):
            text = '\n'.join(rounds[k:])
            token_count = len(self.tokenizer(text, add_special_tokens=False)['input_ids'])
            if token_count < self.max_length - 10:
                break
        else:
            text = '\n'.join(rounds)
        
        return text
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = self._format_conversation(row)
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
        }

## 2. Load Model + Adapter

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(CFG.model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=CFG.use_4bit,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load base model
base_model = AutoModelForSequenceClassification.from_pretrained(
    CFG.model_path,
    num_labels=CFG.num_labels,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
base_model.config.pad_token_id = tokenizer.pad_token_id

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, CFG.adapter_path)
model.eval()
print('Model + adapter loaded successfully')

## 3. Run Inference

In [ ]:
def predict(model, dataset, batch_size=1):
    """Run inference and return softmax probabilities."""
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_probs = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Predicting'):
            input_ids = batch['input_ids'].to(model.device)
            attention_mask = batch['attention_mask'].to(model.device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
    
    return np.concatenate(all_probs, axis=0)

In [ ]:
# Normal prediction
print('Running normal inference...')
dataset_normal = PreferenceDataset(test_df, tokenizer, CFG.max_length, swap=False)
probs_normal = predict(model, dataset_normal, CFG.batch_size)
print(f'Normal preds shape: {probs_normal.shape}')

if CFG.use_tta:
    # TTA: swap responses, then swap prediction columns back
    print('Running TTA (swapped responses)...')
    dataset_swap = PreferenceDataset(test_df, tokenizer, CFG.max_length, swap=True)
    probs_swap = predict(model, dataset_swap, CFG.batch_size)
    
    # After swapping inputs, model_a prediction becomes model_b and vice versa
    # So swap columns 0 and 1 back, keep column 2 (tie) as is
    probs_swap_corrected = probs_swap[:, [1, 0, 2]]
    
    # Average
    final_probs = (probs_normal + probs_swap_corrected) / 2
    print(f'TTA averaged. Shape: {final_probs.shape}')
else:
    final_probs = probs_normal

# Sanity check: probabilities should sum to ~1
row_sums = final_probs.sum(axis=1)
print(f'Row sum stats: min={row_sums.min():.4f}, max={row_sums.max():.4f}, mean={row_sums.mean():.4f}')

## 4. Generate Submission

In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'winner_model_a': final_probs[:, 0],
    'winner_model_b': final_probs[:, 1],
    'winner_tie': final_probs[:, 2],
})

# Ensure no extreme probabilities (clip for numerical stability)
for col in ['winner_model_a', 'winner_model_b', 'winner_tie']:
    submission[col] = submission[col].clip(1e-7, 1 - 1e-7)

# Renormalize after clipping
prob_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
row_sums = submission[prob_cols].sum(axis=1)
for col in prob_cols:
    submission[col] = submission[col] / row_sums

# Save
submission.to_csv('submission.csv', index=False)
print(f'Submission saved: {submission.shape}')
print(submission.head(10))
print(f'\nPrediction stats:')
print(submission[prob_cols].describe())

In [ ]:
# Final validation
assert submission.shape[0] == len(test_df), 'Row count mismatch!'
assert set(submission['id']) == set(test_df['id']), 'ID mismatch!'
assert all(submission[prob_cols].sum(axis=1).between(0.999, 1.001)), 'Probs dont sum to 1!'
assert not submission.isnull().any().any(), 'NaN values found!'
print('All checks passed! Ready to submit.')